# EnSurv on SUPPORT: three-fold cross-validation and diagnostics

This public example trains **the EnSurv package** on the SUPPORT cohort,
displays all evaluation metrics, and plots training and held-out patient diagnostics.

| Task | Imported implementation |
|---|---|
| Encoding | `ensurv.preprocessing.preprocess_df` |
| Model fitting and prediction | `ensurv.model.EnSurv` |
| Every evaluation metric | `ensurv.evaluation.compute_all_metrics` |
| Training plots and time-specific calibration | `ensurv.utils` |
| Population calibration and individual densities | `ensurv.diagnostics.EnSurvDiagnostics` |

SUPPORT's endpoint is **death**. Follow-up is measured
in days. Most physiological assessments are from the third study day; this is
an illustrative prediction exercise using those recorded assessments.

Sources: [SUPPORT data and variable descriptions](https://hbiostat.org/data/repo/supportdesc),
[original data repository](https://hbiostat.org/data/).

## 1. Install EnSurv and import its functions

After the PyPI release, install EnSurv in the environment that runs your notebook kernel:

```bash
python -m pip install "ensurv[workflow]" jupyterlab
```

From a source checkout, replace `ensurv[workflow]` with `.[workflow]` and run the install command at the repository root.

In [ ]:
%matplotlib inline
from pathlib import Path
from zipfile import ZipFile
from urllib.request import Request, urlopen
import json
import pickle
import time
from importlib.metadata import version

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "ensurv").is_dir()),
    Path.cwd(),
)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from ensurv.model import EnSurv
from ensurv.preprocessing import preprocess_df
from ensurv.evaluation import compute_all_metrics
from ensurv.diagnostics import EnSurvDiagnostics
from ensurv.utils import (
    plot_training_dashboard,
    plot_loss_decomposition,
    plot_gradient_flow_heatmap,
    plot_feature_importance_evolution,
    plot_weight_norm_evolution,
    plot_time_calibration,
    plot_pooled_calibration,
)

versions = {name: version(name) for name in
            ["ensurv", "torch", "numpy", "pandas", "scikit-learn",
             "SurvivalEVAL", "lifelines", "matplotlib"]}
display(pd.Series(versions, name="Version").to_frame())
print("Model:", EnSurv.__module__)
print("Metrics:", compute_all_metrics.__module__)
print("Diagnostics:", EnSurvDiagnostics.__module__)

## 2. Configuration and reproducibility

We use shuffled, event-stratified **three-fold CV with
seed 42**, and pass **`random_state=42` to every model**. The existing evaluator
uses its own fixed prediction seed 42 and 2,000 draws without any overrides.

The example uses two **64-unit** hidden layers. The concatenated generator,
uniform noise, direct linear pathway, dropout, and batch normalization follow
the CV setting as the UNOS analysis. The paper's pre-additive generator is also available
through the same class by changing `GENERATOR` to `"pre_additive"`.

`N_PATIENTS=None` uses the entire cohort. For a quick practice run, set it to
600 and reduce `EPOCHS` to 5. These are demonstration settings, not tuned
SUPPORT benchmark results. Hardware and dependency versions can affect exact
floating-point reproducibility.

In [ ]:
SEED = 42
N_SPLITS = 3
N_PATIENTS = None       # None = full SUPPORT cohort; e.g. 600 for practice.
EPOCHS = 50
PATIENCE = 10
HIDDEN_DIMS = [64, 64]
GENERATOR = "concatenated"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

DATA_DIR = PROJECT_ROOT / "data" / "support"
OUTPUT_DIR = PROJECT_ROOT / "support_cv_outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_KWARGS = dict(
    latent_dim=None,
    num_layer=2,
    generator=GENERATOR,
    hidden_dims=HIDDEN_DIMS,
    apply_linear=True,
    dropout=0.1,
    noise_dist="uniform",
    noise_std=1.0,
    batch_normalization=True,
    random_state=SEED,
    tau="max_event",
    time_scale=1.0,
)
FIT_KWARGS = dict(
    epochs=EPOCHS,
    m=5,
    B=20,
    verbose=True,
    patience=PATIENCE,
    track_diagnostics=True,
    diag_every=1,
    lambda_energy=0.5,
    lr=1e-3,
    km_batch_size=256,
    batch_size=4096,
    l1_reg=1e-3,
    domain_constraints=None,  # UNOS covariate constraints do not apply here.
    noise_mode="fixed",
)
print(f"Device: {DEVICE}; folds: {N_SPLITS}; seed: {SEED}")

## 3. Load SUPPORT and select covariates

Download once, then reuse the cached public CSV. To run offline, put
`support2.csv` in `data/support/` before running this cell.

Use 14 recorded demographic, disease, comorbidity, and physiological
covariates. The explicit feature list excludes outcome fields, hospital costs,
discharge length of stay, two-month disability outcomes, and pre-existing
survival predictions. `death=1` marks an observed death; `death=0` marks
right censoring. Only positive observed follow-up times enter the model.

Data obtained from https://hbiostat.org/data courtesy of the Vanderbilt
University Department of Biostatistics. Original study reference:
Knaus WA, Harrell FE, Lynn J et al. (1995), *The SUPPORT prognostic model:
Objective estimates of survival for seriously ill hospitalized adults*,
Annals of Internal Medicine 122:191–203.

In [ ]:
SUPPORT_URL = "https://hbiostat.org/data/repo/support2csv.zip"
csv_path = DATA_DIR / "support2.csv"
zip_path = DATA_DIR / "support2csv.zip"

if not csv_path.exists():
    if not zip_path.exists():
        request = Request(SUPPORT_URL, headers={"User-Agent": "EnSurv-public-example"})
        with urlopen(request, timeout=60) as response:
            zip_path.write_bytes(response.read())
    with ZipFile(zip_path) as archive:
        csv_members = [n for n in archive.namelist()
                       if Path(n).name.lower() == "support2.csv"]
        if len(csv_members) != 1:
            raise ValueError("Expected exactly one support2.csv in the download.")
        csv_path.write_bytes(archive.read(csv_members[0]))

raw = pd.read_csv(csv_path)
FEATURE_COLS = [
    "age", "sex", "dzclass", "num.co", "diabetes", "dementia", "ca",
    "meanbp", "hrt", "resp", "temp", "wblc", "sod", "crea",
]
NOMINAL_COLS = ["sex", "dzclass", "ca"]
required_cols = FEATURE_COLS + ["death", "d.time"]
missing_cols = sorted(set(required_cols) - set(raw.columns))
if missing_cols:
    raise ValueError(f"Missing SUPPORT columns: {missing_cols}")

cohort = raw.loc[
    raw["d.time"].notna() & (raw["d.time"] > 0) & raw["death"].isin([0, 1]),
    required_cols,
].copy()
# Retain original row positions as patient IDs for diagnostic tables.
if N_PATIENTS is not None and N_PATIENTS < len(cohort):
    cohort = cohort.sample(n=N_PATIENTS, random_state=SEED).sort_index()
patient_ids = cohort.index.to_numpy()

durations = cohort["d.time"].to_numpy(dtype=np.float64)
events = cohort["death"].to_numpy(dtype=np.int64)
X_raw = cohort[FEATURE_COLS].copy()
assert np.isfinite(durations).all() and (durations > 0).all()

display(pd.DataFrame({
    "Patients": [len(cohort)],
    "Covariates before encoding": [len(FEATURE_COLS)],
    "Observed deaths": [int(events.sum())],
    "Censored": [int((events == 0).sum())],
    "Maximum follow-up (days)": [float(durations.max())],
}))
display(X_raw.head())
display(X_raw.isna().sum().rename("Missing values").to_frame())

## 4. Prepare each fold with training-only statistics

The short SUPPORT adapter below learns numeric medians and categorical modes
from the training rows. Unseen test categories map to that feature's training
mode. The imported `preprocess_df` then encodes the already filled frames
together so they share the **training-derived** dummy vocabulary and reference
levels. It sees no missing values at this stage and fits no imputation model.
`StandardScaler` is fitted only on training rows; binary and dummy columns
keep their coding.

Time scaling matches the earlier evaluator's contract: fit on
(Y / y_{max}), where (y_{max}) is the maximum training follow-up.
Pass original **day-scale** outcomes and `y_max` to `compute_all_metrics`.
`tau="max_event"` selects the largest observed **training-fold death time**;
its day-scale value is `model.tau_ * y_max`. Do not also set the model's
internal `time_scale` to `y_max`, which would scale times twice.

In [ ]:
def prepare_support_fold(train_idx, test_idx):
    """Adapt SUPPORT using training-only fills and imported feature encoding."""
    train = X_raw.iloc[train_idx].copy()
    test = X_raw.iloc[test_idx].copy()
    numeric_cols = [c for c in FEATURE_COLS if c not in NOMINAL_COLS]

    medians = train[numeric_cols].median()
    if medians.isna().any():
        raise ValueError("A training-fold numeric feature has no observed values.")
    train[numeric_cols] = train[numeric_cols].fillna(medians)
    test[numeric_cols] = test[numeric_cols].fillna(medians)

    category_modes, category_levels = {}, {}
    for column in NOMINAL_COLS:
        observed = train[column].dropna().astype(str)
        if observed.empty:
            raise ValueError(f"No observed training values for {column}.")
        mode = observed.mode().iloc[0]
        levels = sorted(observed.unique().tolist())
        category_modes[column] = mode
        category_levels[column] = levels
        train[column] = train[column].fillna(mode).astype(str)
        test[column] = test[column].fillna(mode).astype(str)
        test[column] = test[column].where(test[column].isin(levels), mode)

    # Encoding only: every fill above came from the training fold.
    combined = pd.concat([train, test], ignore_index=True)
    combined["d.time"] = np.concatenate([durations[train_idx], durations[test_idx]])
    combined["death"] = np.concatenate([events[train_idx], events[test_idx]])
    X_encoded, y_encoded, e_encoded, _, _ = preprocess_df(
        combined, duration_col="d.time", event_col="death",
        nominal_cols=NOMINAL_COLS,
    )
    assert np.array_equal(y_encoded, combined["d.time"].to_numpy())
    assert np.array_equal(e_encoded, combined["death"].to_numpy())

    # Stable column order despite set-based category ordering in preprocess_df.
    X_encoded = X_encoded.reindex(sorted(X_encoded.columns), axis=1).astype(np.float32)
    train_encoded = X_encoded.iloc[:len(train_idx)].copy()
    test_encoded = X_encoded.iloc[len(train_idx):].copy()
    binary_cols = [c for c in train_encoded.columns
                   if set(train_encoded[c].unique()).issubset({0, 1, 0.0, 1.0})]
    cols_to_scale = [c for c in train_encoded.columns
                     if c not in binary_cols
                     and not c.startswith(tuple(NOMINAL_COLS))]
    scaler = StandardScaler()
    if cols_to_scale:
        train_encoded[cols_to_scale] = scaler.fit_transform(
            train_encoded[cols_to_scale]).astype(np.float32)
        test_encoded[cols_to_scale] = scaler.transform(
            test_encoded[cols_to_scale]).astype(np.float32)

    X_train = train_encoded.to_numpy(dtype=np.float32)
    X_test = test_encoded.to_numpy(dtype=np.float32)
    assert np.isfinite(X_train).all() and np.isfinite(X_test).all()
    preprocessing = dict(
        numeric_medians=medians.to_dict(),
        category_modes=category_modes,
        category_levels=category_levels,
        columns_to_scale=cols_to_scale,
        scaler=scaler,
        feature_names=train_encoded.columns.tolist(),
    )
    return X_train, X_test, preprocessing

## 5. Fit three models and evaluate each held-out fold

The loop calls `EnSurv.fit` and **only `compute_all_metrics`** for evaluation.
The evaluator returns `(metrics, times_grid, S_pred)`. Keep those outputs
with their own fold's model, data, feature order, and time scale so subsequent
plots always use a matched context. Early stopping monitors training loss,
as implemented in the existing model; no test outcomes guide training.

Each fold saves metrics, predictions/outcomes, weights, preprocessing, and
recorded history. All three models remain in `fold_results` for diagnostics.

In [ ]:
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_results = []
metric_rows = []
fold_info_rows = []
cv_start = time.perf_counter()

for fold, (train_idx, test_idx) in enumerate(splitter.split(X_raw, events), start=1):
    print(f"\n--- Fold {fold}/{N_SPLITS} ---")
    X_train, X_test, preprocessing = prepare_support_fold(train_idx, test_idx)
    feature_names = preprocessing["feature_names"]
    y_train, y_test = durations[train_idx], durations[test_idx]
    e_train, e_test = events[train_idx], events[test_idx]
    y_max = float(y_train.max())

    model = EnSurv(device=DEVICE, **MODEL_KWARGS)
    fold_start = time.perf_counter()
    model.fit(X_train, y_train / y_max, e_train,
              feature_names=feature_names, **FIT_KWARGS)
    fit_seconds = time.perf_counter() - fold_start

    # Every metric is computed by the unchanged evaluation.py function.
    metrics, times_grid, S_pred = compute_all_metrics(
        model, y_train, e_train, X_test, y_test, e_test, y_max=y_max,
    )
    metric_rows.append({"Fold": fold, **metrics})
    context = dict(
        fold=fold, model=model, history=model.history,
        train_idx=train_idx, test_idx=test_idx,
        patient_ids=patient_ids[test_idx],
        X_train=X_train, X_test=X_test,
        y_train=y_train, y_test=y_test, e_train=e_train, e_test=e_test,
        y_max=y_max, tau_days=float(model.tau_ * y_max),
        feature_names=feature_names, preprocessing=preprocessing,
        metrics=metrics, times_grid=times_grid, S_pred=S_pred,
    )
    fold_results.append(context)
    fold_info_rows.append(dict(
        Fold=fold, Train_N=len(train_idx), Test_N=len(test_idx),
        Train_deaths=int(e_train.sum()), Test_deaths=int(e_test.sum()),
        Encoded_features=len(feature_names), Epochs=len(model.history["epoch"]),
        Time_divisor_days=y_max, Tau_days=context["tau_days"],
        Fit_seconds=fit_seconds,
    ))

    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(exist_ok=True)
    pd.DataFrame([metrics]).to_csv(fold_dir / "metrics.csv", index=False)
    np.savez_compressed(
        fold_dir / "predictions.npz", S_pred=S_pred, times_grid=times_grid,
        y_train=y_train, e_train=e_train, y_test=y_test, e_test=e_test,
        train_idx=train_idx, test_idx=test_idx,
        patient_ids=patient_ids[test_idx],
        y_max=y_max, tau_days=context["tau_days"],
    )
    torch.save({
        "state_dict": model.model.state_dict(),
        "model_kwargs": MODEL_KWARGS,
        "fit_kwargs": FIT_KWARGS,
        "input_dim": model.input_dim,
        "feature_names": feature_names,
        "y_max": y_max,
        "tau_model_units": model.tau_,
        "time_scale_model_units": model.time_scale_,
    }, fold_dir / "model_checkpoint.pt")
    with (fold_dir / "history.pkl").open("wb") as file:
        pickle.dump(model.history, file)
    with (fold_dir / "preprocessing.pkl").open("wb") as file:
        pickle.dump(preprocessing, file)

    display(pd.Series(metrics, name=f"Fold {fold}").to_frame().style.format("{:.4g}"))

fold_info = pd.DataFrame(fold_info_rows).set_index("Fold")
fold_metrics = pd.DataFrame(metric_rows).set_index("Fold")
print(f"\nThree-fold CV completed in {time.perf_counter() - cv_start:.1f} seconds.")
display(fold_info.round(3))

## 6. Display every metric and its across-fold summary

The summary reports arithmetic means, sample standard deviations (`ddof=1`),
and the number of nonmissing folds. It summarizes the evaluator's results;
it does not recompute any survival metric. NaNs remain visible.

- Concordance, AUC, and AUPRC: larger values indicate better discrimination.
- MAE/error, Brier/IBS, and KM-calibration discrepancies: smaller is better.
  MAE/error values here use **days**.

**Horizon interpretation:** the existing evaluator queries the raw generator
with `allow_extrapolation=True` over the full follow-up grid. Values at or
beyond a fold's `Tau_days` are working-model extrapolations.

In [ ]:
metric_summary = pd.DataFrame({
    "Mean": fold_metrics.mean(axis=0),
    "Std": fold_metrics.std(axis=0, ddof=1),
    "Valid_folds": fold_metrics.count(axis=0),
})
metric_summary["Mean ± Std"] = [
    f"{mean:.4g} ± {std:.4g}" if pd.notna(mean) and pd.notna(std) else "NaN"
    for mean, std in zip(metric_summary["Mean"], metric_summary["Std"])
]
print("Held-out fold metrics:")
display(fold_metrics.T.style.format("{:.4g}", na_rep="NaN"))
print("Across-fold summaries:")
display(metric_summary)

fold_metrics.to_csv(OUTPUT_DIR / "EnSurv_SUPPORT_3Fold_Metrics.csv")
metric_summary.to_csv(OUTPUT_DIR / "EnSurv_SUPPORT_3Fold_Summary.csv")
fold_info.to_csv(OUTPUT_DIR / "EnSurv_SUPPORT_Fold_Info.csv")
run_config = dict(
    seed=SEED, n_splits=N_SPLITS, n_patients=len(cohort),
    dataset_url=SUPPORT_URL, selected_covariates=FEATURE_COLS,
    model=MODEL_KWARGS, fit=FIT_KWARGS, device=str(DEVICE), versions=versions,
)
(OUTPUT_DIR / "run_configuration.json").write_text(
    json.dumps(run_config, indent=2), encoding="utf-8",
)

## 7. Training diagnostics for all three folds

Call the imported dashboard with each model's recorded `history`. The panels
show loss/components, gradient flow, structural feature-importance evolution,
and linear/nonlinear weight norms. The recorded path can extend past the
epoch whose weights were restored by early stopping.

In the existing history, `event_loss` also includes outcomes known through
follow-up to the restriction horizon. `censor_loss` is the **pooled KM
completion term**.

In [ ]:
for result in fold_results:
    fold = result["fold"]
    fig, _ = plot_training_dashboard(
        result["history"], feature_names=result["feature_names"], top_k=6,
        save_path=OUTPUT_DIR / f"fold_{fold}" / "training_dashboard.pdf",
        suptitle=f"SUPPORT: EnSurv training diagnostics — fold {fold}",
    )
    plt.show()
    plt.close(fig)

### Inspect individual training panels for one fold

Change `DIAGNOSTIC_FOLD` to 1, 2, or 3. This choice affects the remaining
diagnostic cells and never selects a fold based on its evaluation performance.

In [ ]:
DIAGNOSTIC_FOLD = 1
if DIAGNOSTIC_FOLD not in range(1, N_SPLITS + 1):
    raise ValueError("DIAGNOSTIC_FOLD must be 1, 2, or 3.")
diagnostic_result = fold_results[DIAGNOSTIC_FOLD - 1]
history = diagnostic_result["history"]
feature_names = diagnostic_result["feature_names"]
diagnostic_dir = OUTPUT_DIR / f"fold_{DIAGNOSTIC_FOLD}"

fig, axes = plot_loss_decomposition(history)
# Correct the inherited legend wording for this example; calculations are unchanged.
axes[1].lines[0].set_label("Observed/restricted outcome term")
axes[1].lines[1].set_label("Pooled KM completion term")
axes[1].legend(loc="best")
fig.savefig(diagnostic_dir / "loss_components.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

for plotter, kwargs, filename in [
    (plot_gradient_flow_heatmap, {}, "gradient_flow.pdf"),
    (plot_feature_importance_evolution,
     {"feature_names": feature_names, "top_k": 8}, "importance_evolution.pdf"),
    (plot_weight_norm_evolution, {}, "weight_norms.pdf"),
]:
    fig, _ = plotter(history, save_path=diagnostic_dir / filename, **kwargs)
    plt.show()
    plt.close(fig)

## 8. Inspect the fitted feature summaries

The model's `explain` method provides raw linear coefficients and
nonlinear structural contribution norms in the fitted encoded feature order.
Do not interpret these as causal effects or day-scale regression coefficients.

In [ ]:
for result in fold_results:
    explanations = result["model"].explain(feature_names=result["feature_names"])
    print(f"Fold {result['fold']}: largest structural feature summaries")
    display(explanations.head(10))
    explanations.to_csv(
        OUTPUT_DIR / f"fold_{result['fold']}" / "feature_explanations.csv", index=False,
    )

print("Saved results in:", OUTPUT_DIR.name)
display(pd.DataFrame({
    "Output": ["Fold metrics and mean/std summary", "Fold sizes, epochs, and horizons",
               "Run settings and dependency versions", "Per-fold predictions and outcomes",
               "Weights, preprocessing, history, and feature summaries",
               "Training and post-training figures"],
    "Location": ["EnSurv_SUPPORT_3Fold_*.csv", "EnSurv_SUPPORT_Fold_Info.csv",
                 "run_configuration.json", "fold_*/predictions.npz",
                 "fold_*/", "fold_*/*.pdf and pooled_oof_calibration.pdf"],
}))

### Reuse this example with another public survival dataset

Replace only the dataset-loading cell, covariate list, and dataset-specific
categorical declarations. Keep the imported training, metric, and plotting
calls. Outcomes passed to the evaluator must remain in **days**, because its
clinical horizons are fixed at 365.25 and 1826.25 days. A different endpoint
does not change the fact that `events=1` must denote an observed failure.